In [1]:
from __future__ import print_function, division
from future.utils import iteritems
from builtins import range
from gensim.models import KeyedVectors
import numpy as np

In [2]:
# 1. Load the model directly from the file
local_path = "./offline_material/embedding/glove-twitter-25.kv"
glove_model = KeyedVectors.load(local_path)


In [8]:
def dist1(a, b):
    return np.linalg.norm(a - b)

def dist2(a, b):
    return 1 - a.dot(b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [9]:
# pick a distance type
dist, metric = dist2, 'cosine'
# dist, metric = dist1, 'euclidean'

In [30]:
## more intuitive
def find_analogies(w1, w2, w3, word2vec):
    for w in (w1, w2, w3):
        if w not in word2vec:
            print(f'not in dictionary {w}')
            return
        
    king = word2vec[w1]
    man = word2vec[w2]
    woman = word2vec[w3]
    v0 = king - man + woman

    min_dist = float('inf')
    best_word = ''
    for word, v1 in iteritems(word2vec):
        if word not in (w1, w2, w3):
            d = dist(v0, v1)
            if d < min_dist:
                min_dist = d
                best_word = word
    print(w1, '-', w2, '=', best_word, '-', w3)

In [25]:
similar_words = glove_model.most_similar('movie', topn=3)
print(similar_words) 

[('episode', 0.9394966959953308), ('song', 0.9384620189666748), ('songs', 0.9332047700881958)]


In [ ]:
print('Loading word vectors...')
word2vec = {}
embeding = []
idx2word = []


# 2. Use it
print(glove_model['computer'].shape)
word2vec = {word: glove_model[word] for word in glove_model.key_to_index.keys()}


# embeding = np.array(embeding)
# V, D = embeding.shape
# word2vec = glove_model.get_vector('hello')


# find_analogies('king', 'man', 'woman')
# find_analogies('france', 'paris', 'london')
# find_analogies('france', 'paris', 'rome')

Loading word vectors...
(25,)


In [39]:
find_analogies('england', 'london', 'iran', word2vec)


england - london = sanctions - iran


In [61]:
score = glove_model.similarity('teacher', 'house')
print(score) # Might be something like 0.85


0.77496094


In [26]:
from gensim.models import KeyedVectors

# 1. Load your model
local_path = "./offline_material/embedding/glove-twitter-25.kv"
glove_model = KeyedVectors.load(local_path) # Or whatever your local path is

# 2. Define your analogy function
def find_analogy(model, word_a, word_b, word_c):
    """
    Solves: word_a is to word_b as word_c is to ?
    Math: ? = word_b - word_a + word_c
    """
    try:
        # positive = words we ADD (word_b, word_c)
        # negative = words we SUBTRACT (word_a)
        result = model.most_similar(positive=[word_b, word_c], negative=[word_a], topn=1)
        
        # result looks like: [('queen', 0.85)]
        best_word, similarity_score = result[0]
        
        print(f"'{word_a}' is to '{word_b}' as '{word_c}' is to '{best_word}'")
        
    except KeyError as e:
        print(f"Error: One of the words is not in the model's vocabulary: {e}")

# 3. Test some classic analogies!

# Example 1: Gender & Royalty
# Math: King - Man + Woman = ?
find_analogy(glove_model, word_a='man', word_b='king', word_c='woman')

# Example 2: Capitals & Countries
# Math: Paris - France + Italy = ?
find_analogy(glove_model, word_a='france', word_b='paris', word_c='italy')

# Example 3: Grammar / Tenses
# Math: Walked - Walk + Swim = ?
find_analogy(glove_model, word_a='walk', word_b='walked', word_c='swim')

'man' is to 'king' as 'woman' is to 'meets'
'france' is to 'paris' as 'italy' is to 'brazil'
'walk' is to 'walked' as 'swim' is to 'smashed'


## pretrained glove 25

In [6]:
def check(word):
    if word in glove_model:
        return glove_model[word]

In [8]:
check('jkjklibnln')

In [29]:
def instance(w1, w2, w3, glove_model):
    vec1 = check(w1)
    vec2 = check(w2)
    vec3 = check(w3)
    vec = vec1 - vec2 + vec3
    word = glove_model.similar_by_vector(vec, topn=3)
    return word

In [30]:
vec = instance('spain', 'madrid', 'tehran', glove_model)
vec

[('tehran', 0.872529149055481),
 ('copts', 0.8167942762374878),
 ('путана', 0.8016719818115234)]

In [21]:
type(glove_model)

gensim.models.keyedvectors.KeyedVectors

In [22]:
# Get the unique integer ID for the word
word_id = glove_model.key_to_index["apple"]
print(f"ID for 'apple': {word_id}")

ID for 'apple': 1881


### 1 out of 4 words

In [33]:
a = glove_model.distance('iran', 'apple')
a

np.float32(0.41279334)

In [54]:
word_list = ['car', 'london', 'paris', 'tehran', 'rome']
a = glove_model.distances(word_or_vector=word_list[0], other_words=word_list[1:])
a

array([0.29510438, 0.331618  , 0.7475809 , 0.41827118], dtype=float32)

In [ ]:
def one_out_of_four(w1, w2, w3, w4, glove_model):
    glove_model.distance(w1, w2)

## word anology

In [55]:
def cal_dist(word1, word2):
    return 1 - word1.dot(word2) / (np.linalg.norm(word1) * np.linalg.norm(word2))

In [56]:
def word_anology(w1: str, w2: str, w3: str, word2vec: dict):
    for word in [w1, w2, w3]:
        if word not in word2vec:
            print(f'The word {word} is not in the word2vec dictioinary.')
            return
    vec1 = word2vec[w1]
    vec2 = word2vec[w2]
    vec3 = word2vec[w3]
    vec4 = vec1 - vec2 + vec3

    min_distance = float('inf')
    best_word = ''
    for word, vec in word2vec.items():
        if word not in (w1, w2, w3):
            dist = cal_dist(vec, vec4)
            if dist < min_distance:
                min_distance = dist
                best_word = word
    
    print(best_word)

In [ ]:
def nearest_neighbor(word: str, n: int, word2vec: dict):
    if word not in word2vec:
        print(f"The word {word} is not in the dictionary.")
        return
    vec = word2vec[word]
    distances = pairwise_distances(vec.reshape(1, D), embedding, metric=metric).reshape(V)
    idxs = distances.argsort()[1:n+1]
    for idx in idxs:
        print(f'\t {idx2word[idx]}')

In [ ]:
def read_pre_trained_wordvec():
    embedding = []
    word2vec = {}
    idx2word = []
    path = ''
    with open(path) as f:
        for line in f:
            values = line.split()
            word = values[0]
            vector = np.array(values[1:], dtype='float32')
            word2vec[word] = vector
            embedding.append(vector)
            idx2word.append(word)
    print(f'Found {len(word2vec)} word vectors.')
    embedding = np.array(embedding)
    V, D = embedding.shape

# pre-trained word2vec

- The words of vectors have a vocabulary size of three million whereas the glove vectors only use a vocabulary size of 400,000 - few gigabyte
- The word vectors are provided in a binary format. This allows the file to be smaller in size but it also means you have to figure out the structure of the file if you want to be able to parse it. That's not ideal for us because it doesn't allow us to think about how analogies and neighbors are calculated.

In [58]:
from gensim.models import KeyedVectors

In [60]:
# 1. Load the model directly from the file
local_path = "./offline_material/embedding/glove-twitter-25.kv"
word_vectors = KeyedVectors.load(local_path)

type(word_vectors)
# word_vectors = KeyedVectors.load_word2vec_format(local_path, binary=True)

gensim.models.keyedvectors.KeyedVectors

In [61]:
def find_analogies(w1, w2, w3, word_vectors: KeyedVectors):
    r = word_vectors.most_similar(positive=[w1, w3], negative=[w2], topn=1)
    print(f'{w1} - {w2} = {r[0][0]} - {w3}')

In [62]:
def nearest_neighbor(w: str, word_vectors: KeyedVectors):
    r = word_vectors.most_similar(positive=[w], topn=5)
    print(f'neighbors of {w}:')
    for word, score in r:
        print(word, score)

In [64]:
find_analogies('king', 'man', 'woman', word_vectors)
find_analogies('france', 'paris', 'london', word_vectors)

king - man = meets - woman
france - paris = canada - london


In [74]:
nearest_neighbor('king', word_vectors)
nearest_neighbor('sex', word_vectors)

neighbors of king:
prince 0.9337409734725952
queen 0.9202421307563782
aka 0.9176921844482422
lady 0.9163240790367126
jack 0.9147354364395142
neighbors of sex:
kids 0.8704328536987305
porn 0.8667966723442078
self 0.8664330840110779
child 0.8578428030014038
person 0.8556257486343384
